# Start by reading in a physics state:

In [37]:
import xarray as xr
import ndsl.dsl.gt4py_utils as gt_utils
from ndsl import GridSizer, Quantity, QuantityFactory, TileCommunicator, TilePartitioner, NullComm, SubtileGridSizer
import ndsl.constants as constants

from pySHiELD.physics_state import PhysicsState
from pySHiELD._config import PHYSICS_PACKAGES
import numpy as np

In [10]:
ds = xr.open_dataset("RESTART/restart_physics_state_0.nc")
ds2 = xr.open_dataset("RESTART/restart_dycore_state_0.nc")

In [3]:
schemes = PHYSICS_PACKAGES

In [4]:
nx_tile=20
ny_tile=20
nz=79
n_halo=3

In [5]:
rank = 0

comm = NullComm(rank, 1)
communicator = TileCommunicator.from_layout(comm=comm, layout=(1,1))

sizer = SubtileGridSizer.from_tile_params(
    nx_tile=nx_tile,
    ny_tile=ny_tile,
    nz=nz,
    n_halo=n_halo,
    extra_dim_lengths={},
    layout=(1,1),
    tile_partitioner=communicator.partitioner.tile,
    tile_rank=communicator.tile.rank,
)
quantity_factory = QuantityFactory.from_backend(
    sizer, backend="numpy"
)

In [6]:
state = PhysicsState.init_zeros(quantity_factory, schemes)

In [24]:
pkz = ds2.pkz.values
pe = ds2.pe.values
pk = ds2.pk.values

In [23]:
p_lay = pkz[:,:,::-1]** (1./constants.KAPPA)

# Now we need certain things from the physics state:
 - level pressure in Pa
 - layer pressure in Pa
 - level temperature in K
 - layer temperature in K
 - surface temperature in K
 - water vapor mixing ratio in kg/kg
 - ozone mixing ratio in kg/kg

In [ ]:
print(ds.variables)

In [27]:
ds.delp.shape

(27, 27, 80)

In [82]:
prsi = ds.prsi.values[3:-4,3:-4,:] # level pressure
pt = ds.pt.values[3:-4,3:-4,:-1] # Layer temperature
delp = ds.delp.values[3:-4,3:-4,:-1]
delz = -1.* ds.delz.values[3:-4,3:-4,:-1]
dz = -1.* ds.dz.values[3:-4,3:-4,:-1]
qvap = ds.qvapor.values[3:-4,3:-4,:-1]
qliq = ds.qliquid.values[3:-4,3:-4,:-1]
qice = ds.qice.values[3:-4,3:-4,:-1]

Derive layer mean pressure from interface pressure

In [85]:
prsl = (prsi[:,:,1:] - prsi[:,:,:-1]) / np.log(prsi[:,:,1:] / prsi[:,:,:-1])

Or from delp delz T and Q

In [83]:
# prsl = delp / (constants.GRAV * dz) * constants.RDGAS * pt * (1 + constants.ZVIR * qvap)

In [86]:
prsl

array([[[  451.37975146,   830.08703879,  1245.19882957, ...,
         98937.69113034, 99419.49467629, 99811.11444245],
        [  451.37975146,   830.08703879,  1245.19882957, ...,
         98984.07452286, 99466.15895168, 99857.98743327],
        [  451.37975146,   830.08703879,  1245.19882957, ...,
         98970.52381759, 99452.5261877 , 99844.29369399],
        ...,
        [  451.37975146,   830.08703879,  1245.19882957, ...,
         98982.68148138, 99464.75747439, 99856.5796876 ],
        [  451.37975146,   830.08703879,  1245.19882957, ...,
         98995.82512649, 99477.98071317, 99869.86206999],
        [  451.37975146,   830.08703879,  1245.19882957, ...,
         98947.19874714, 99429.05986815, 99820.72241658]],

       [[  451.37975146,   830.08703879,  1245.19882957, ...,
         98912.53133844, 99394.1825248 , 99785.68907724],
        [  451.37975146,   830.08703879,  1245.19882957, ...,
         98956.06884188, 99437.98367732, 99829.68613929],
        [  451.37975146, 

In [45]:
ptlev = np.zeros_like(prsi)
ptlev[:,:,1:-1] = pt[:,:,:-1] + (pt[:,:,1:] - pt[:,:,:-1]) * (np.log(prsi[:,:,1:-1]) - np.log(prsl[:,:,:-1])) / (np.log(prsl[:,:,1:]) - np.log(prsl[:,:,:-1]))
ptlev[:,:,-1] = pt[:,:,-2]
ptlev[:,:,0] = pt[:,:,0]

In [48]:
qo3mr = ds.qo3mr.values
tskin = ptlev[:,:,0]

# Time to get the radiation going

In [ ]:
from pyrte_rrtmgp import rrtmgp_cloud_optics, rrtmgp_gas_optics
from pyrte_rrtmgp.data_types import (
    CloudOpticsFiles,
    GasOpticsFiles,
    OpticsProblemTypes,
)
from pyrte_rrtmgp.rte_solver import rte_solve
from pyrte_rrtmgp.examples import (
    compute_RCE_clouds,
    compute_RCE_profiles,
    ALLSKY_EXAMPLES,
    load_example_file,
)